# ADT7420 temperatures on the Grove 16x2 LCD

Reads the three temperatures produced by the `adt7420_axil` core and shows them
on the Grove 16x2 LCD (AiP31068L) that we brought up earlier in this chapter.

This notebook needs the **combined** overlay (`adt7420_float`) whose block
design contains *both* the `adt7420_axil` register core **and** the AXI IIC core
wired to the Grove LCD connector.  In that design:

* the ADT7420 sensor sits on the core's own I2C master, and
* the Grove LCD sits on the AXI IIC core (I2C address `0x3E`).

## `adt7420_axil` register map (read-only, 32-bit)

| Offset | Name         | Contents |
|:------:|:-------------|:---------|
| `0x00` | `TEMP_INST`  | instantaneous measurement.  `degC = value/128` |
| `0x04` | `TEMP_AVG`   | 16-sample integer average.  `degC = value/128` |
| `0x08` | `STATUS`     | `[0]` busy, `[1]` error, `[31:16]` sample counter |
| `0x0C` | `ID`         | `0x00007420` |
| `0x10` | `TEMP_FLT`   | floating-point moving-average filtered temp.  `degC = value/128` |
| `0x14` | `TEMP_FLT_F` | same value converted to Fahrenheit in hardware.  `degF = value/128` |

`TEMP_FLT` / `TEMP_FLT_F` are the output of the `flt_temp` floating-point
pipeline (smoothing done in single-precision FP, then converted back to a Q8.7
fixed-point value), so every temperature register divides by 128.

## LCD layout (16x2)

```
Cur23.5 Avg23.4      <- instantaneous , integer average
Flt23.6C 74.5F       <- FP-filtered degC , FP-filtered degF
```


In [ ]:
from pynq import Overlay, MMIO

# Combined overlay: must contain BOTH adt7420_axil and an AXI IIC core.
# The matching .hwh must sit alongside the .bit.
BITSTREAM = "hw_wrapper.bit"


def find_ip(ol, *, name=None, iptype=None):
    """Return an MMIO window onto the first IP matching name (substring) or type."""
    for ipname, info in ol.ip_dict.items():
        if name and name.lower() in ipname.lower():
            print(f"Using {ipname} @ {hex(info['phys_addr'])}")
            return MMIO(info["phys_addr"], info["addr_range"])
        if iptype and iptype in info.get("type", ""):
            print(f"Using {ipname} ({info['type']}) @ {hex(info['phys_addr'])}")
            return MMIO(info["phys_addr"], info["addr_range"])
    raise RuntimeError(f"No matching IP (name={name}, type={iptype}). "
                       f"Available: {list(ol.ip_dict)}")


ol   = Overlay(BITSTREAM)
temp = find_ip(ol, name="adt7420")            # the register core
iic  = find_ip(ol, iptype="axi_iic")          # drives the Grove LCD

## Grove LCD driver

The same minimal AXI IIC master-transmit driver and AiP31068L bring-up we
developed in `grove_lcd_pynq_final.py`, embedded here so the notebook is
self-contained.

In [ ]:
import time


class AxiIic:
    """Minimal AXI IIC master-transmit driver (dynamic controller mode)."""

    ISR, SOFTR, CR, SR, TX_FIFO = 0x020, 0x040, 0x100, 0x104, 0x108
    CR_EN, CR_TXFIFO_RESET      = 0x01, 0x02
    SR_BB                       = 0x04           # bus busy
    ISR_TX_ERROR                = 0x02           # slave NACK in master-transmit
    START, STOP                 = 0x100, 0x200   # dynamic-mode TX FIFO markers

    def __init__(self, mmio):
        self.mmio = mmio
        self.reset()

    def reset(self):
        self.mmio.write(self.SOFTR, 0x0A)                # soft reset (clears BB)
        time.sleep(0.001)
        self.mmio.write(self.CR, self.CR_TXFIFO_RESET)   # flush TX FIFO
        self.mmio.write(self.CR, self.CR_EN)             # enable

    def _wait_bus_idle(self, timeout=0.2):
        t0 = time.time()
        while self.mmio.read(self.SR) & self.SR_BB:
            if time.time() - t0 > timeout:
                raise TimeoutError("AXI IIC bus stuck busy (check wiring / pull-ups)")

    def write_bytes(self, addr7, data):
        """Write `data` to 7-bit `addr7` in one transaction; raises on NACK."""
        self._wait_bus_idle()
        self.mmio.write(self.ISR, self.mmio.read(self.ISR))     # clear stale flags
        self.mmio.write(self.TX_FIFO, self.START | (addr7 << 1))
        n = len(data)
        for i, b in enumerate(data):
            word = b & 0xFF
            if i == n - 1:
                word |= self.STOP
            self.mmio.write(self.TX_FIFO, word)
        time.sleep(0.001)
        self._wait_bus_idle()
        if self.mmio.read(self.ISR) & self.ISR_TX_ERROR:
            self.reset()
            raise IOError(f"I2C NACK from 0x{addr7:02X}")

In [ ]:
class GroveLCD:
    """Grove 16x2 LCD (AiP31068L, ST7032-compatible) at I2C address 0x3E."""

    ADDR = 0x3E

    def __init__(self, iic, contrast=0x38):
        self.iic = iic
        self._init(contrast)

    def _cmd(self, c):
        self.iic.write_bytes(self.ADDR, [0x80, c])

    def _char(self, c):
        self.iic.write_bytes(self.ADDR, [0x40, c])

    def _init(self, contrast):
        time.sleep(0.05)
        self._cmd(0x38)                                    # 8-bit, 2-line, IS=0
        self._cmd(0x39)                                    # IS=1 (extended set)
        self._cmd(0x14)                                    # OSC / bias (1/5)
        self._cmd(0x70 | (contrast & 0x0F))                # contrast low nibble
        self._cmd(0x50 | 0x04 | ((contrast >> 4) & 0x03))  # booster ON + contrast high
        self._cmd(0x6C)                                    # follower ON
        time.sleep(0.30)                                   # booster/follower settle
        self._cmd(0x38)                                    # back to normal set
        self._cmd(0x0C)                                    # display ON, cursor off
        self.clear()
        self._cmd(0x06)                                    # entry mode: increment

    def clear(self):
        self._cmd(0x01)
        time.sleep(0.002)

    def set_cursor(self, col, row):
        self._cmd(0x80 | (col | (0x40 if row else 0x00)))

    def write(self, text):
        for ch in text:
            self._char(ord(ch))


lcd = GroveLCD(AxiIic(iic), contrast=0x38)
lcd.set_cursor(0, 0); lcd.write("ADT7420 + LCD")
lcd.set_cursor(0, 1); lcd.write("initializing...")
print("LCD ready")

In [ ]:
# Register offsets (byte addresses)
REG_TEMP_INST   = 0x00
REG_TEMP_AVG    = 0x04
REG_STATUS      = 0x08
REG_ID          = 0x0C
REG_TEMP_FLT    = 0x10   # FP-filtered temperature, degC
REG_TEMP_FLT_F  = 0x14   # FP-filtered temperature, degF (computed in HW)


def _signed32(v):
    return v - (1 << 32) if v & 0x8000_0000 else v


def scaled(reg):
    """Read a temperature register and return degrees (degC or degF), value/128."""
    return _signed32(temp.read(reg)) / 128.0


def read_status():
    s = temp.read(REG_STATUS)
    return {"busy": bool(s & 1), "error": bool(s & 2), "samples": (s >> 16) & 0xFFFF}


# Sanity check: ID register should read 0x00007420.
dev_id = temp.read(REG_ID)
print(f"ID     = 0x{dev_id:08X} "
      f"({'OK' if dev_id == 0x0000_7420 else 'unexpected - check overlay/address'})")
st = read_status()
print(f"STATUS = busy={st['busy']}  error={st['error']}  samples={st['samples']}")

In [ ]:
def lcd_show(inst, avg, flt_c, flt_f):
    """Render the three temperatures onto the 16x2 panel."""
    line0 = f"Cur{inst:4.1f} Avg{avg:4.1f}".ljust(16)[:16]
    line1 = f"Flt{flt_c:4.1f}C {flt_f:4.1f}F".ljust(16)[:16]
    lcd.set_cursor(0, 0); lcd.write(line0)
    lcd.set_cursor(0, 1); lcd.write(line1)


# One-shot update
lcd_show(scaled(REG_TEMP_INST), scaled(REG_TEMP_AVG),
         scaled(REG_TEMP_FLT),  scaled(REG_TEMP_FLT_F))
print(f"inst = {scaled(REG_TEMP_INST):6.2f} C")
print(f"avg  = {scaled(REG_TEMP_AVG):6.2f} C")
print(f"flt  = {scaled(REG_TEMP_FLT):6.2f} C  ({scaled(REG_TEMP_FLT_F):6.2f} F)")

## Live update

Refreshes the LCD each time the hardware sample counter advances (a genuinely
new measurement, ~`SAMPLE_HZ` times a second).  Interrupt the kernel (&#9632; /
Ctrl-C) to stop.

In [ ]:
last = -1
try:
    while True:
        st = read_status()
        if st["samples"] != last:                     # new measurement available
            last  = st["samples"]
            inst  = scaled(REG_TEMP_INST)
            avg   = scaled(REG_TEMP_AVG)
            flt_c = scaled(REG_TEMP_FLT)
            flt_f = scaled(REG_TEMP_FLT_F)
            lcd_show(inst, avg, flt_c, flt_f)
        time.sleep(0.05)
except KeyboardInterrupt:
    lcd.clear()
    lcd.write("stopped")
    print("Stopped.")

## Paged view (one reading per screen, with the &deg; symbol)

The static two-line layout above is cramped, so there is no room for the degree
symbol.  This variant instead pages through the readings one at a time, giving
each its own full-width line with `\xdfC` / `\xdfF` units.  On the AiP31068L
character ROM, byte `0xDF` is the degree glyph.

```
Current temp             FP filtered
 23.5*C   74.3*F    ->    23.6*C   74.5*F
```
(the `*` is the degree symbol on the panel).  Each page dwells for `DWELL`
seconds, then advances; `Ctrl-C` to stop.

In [ ]:
DEG = chr(0xDF)   # degree symbol in the AiP31068L / HD44780 character ROM


def fahr(c):
    return c * 9.0 / 5.0 + 32.0


def lcd_page(title, c, f):
    """Show one reading full-width: title on line 0, degC + degF on line 1."""
    line1 = f"{c:5.1f}{DEG}C{f:6.1f}{DEG}F"
    lcd.set_cursor(0, 0); lcd.write(title.ljust(16)[:16])
    lcd.set_cursor(0, 1); lcd.write(line1.ljust(16)[:16])


# The pages: FP-filtered uses the hardware-computed Fahrenheit register; the
# others convert to Fahrenheit in software.
PAGES = [
    ("Current temp", lambda: (lambda c: (c, fahr(c)))(scaled(REG_TEMP_INST))),
    ("Average (16)", lambda: (lambda c: (c, fahr(c)))(scaled(REG_TEMP_AVG))),
    ("FP filtered",  lambda: (scaled(REG_TEMP_FLT), scaled(REG_TEMP_FLT_F))),
]

# One-shot: show the current page set once.
for _title, _reader in PAGES:
    _c, _f = _reader()
    print(f"{_title:14s}: {_c:6.2f}{DEG}C  {_f:6.2f}{DEG}F")
lcd_page(*(("Current temp",) + PAGES[0][1]()))

In [ ]:
DWELL = 2.0   # seconds each page stays on screen

try:
    while True:
        for title, reader in PAGES:
            c, f = reader()
            lcd_page(title, c, f)
            time.sleep(DWELL)
except KeyboardInterrupt:
    lcd.clear()
    lcd.write("stopped")
    print("Stopped.")